# 01 — PyTorch Tensors, Devices, and Shapes

**Overview:** tensor creation, dtype/device inspection, NumPy↔PyTorch conversion, device
detection and movement, and every core tensor operation from Day 1's `02_numpy`, mirrored in
PyTorch and verified numerically identical.

**Required libraries:** `torch`, `numpy`.

Connects to: Day 1's `02_numpy` (nearly identical operations) and `17_deep_learning_architectures`
(the matmuls this notebook benchmarks are exactly what attention needs at scale). Leads to:
`19_pytorch_autograd`.

## 0. Environment

In [1]:
import torch
import numpy as np
print("torch:", torch.__version__)
print("numpy:", np.__version__)
torch.manual_seed(42)


torch: 2.14.1
numpy: 2.5.3


## 1. Creating tensors and inspecting shape/dtype/device

💡 **Key Idea:** a tensor is defined by its values PLUS its shape, dtype, and device -- all
four matter.

In [2]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.randn(2, 3, 4)   # standard normal, shape (2,3,4)

for name, t in [("a", a), ("b", b), ("c", c)]:
    print(f"{name}: shape={tuple(t.shape)}, dtype={t.dtype}, device={t.device}")

assert a.shape == (3,)
assert b.shape == (2, 3)
assert c.shape == (2, 3, 4)
assert a.dtype == torch.float32   # PyTorch's default float dtype


a: shape=(3,), dtype=torch.float32, device=cpu
b: shape=(2, 3), dtype=torch.float32, device=cpu
c: shape=(2, 3, 4), dtype=torch.float32, device=cpu


## 2. NumPy <-> PyTorch conversion

⚠️ **Common Mistake:** converting a NumPy `float64` array without casting to `float32` first --
wastes memory and can silently slow down GPU code, since most deep learning kernels are
optimized for `float32`.

In [3]:
numpy_array = np.random.default_rng(42).normal(size=(3, 3)).astype(np.float32)
tensor_from_numpy = torch.from_numpy(numpy_array)
print("dtype after from_numpy (float32 in, float32 out):", tensor_from_numpy.dtype)

numpy_float64 = np.random.default_rng(42).normal(size=(3, 3))   # NumPy defaults to float64
tensor_float64 = torch.from_numpy(numpy_float64)
print("dtype from a float64 NumPy array (NOT the deep-learning default!):", tensor_float64.dtype)
tensor_float32 = tensor_float64.float()   # explicit cast to the deep-learning default
print("after explicit .float() cast:", tensor_float32.dtype)

back_to_numpy = tensor_from_numpy.numpy()
assert np.allclose(back_to_numpy, numpy_array)
assert tensor_float64.dtype == torch.float64
assert tensor_float32.dtype == torch.float32


dtype after from_numpy (float32 in, float32 out): torch.float32
dtype from a float64 NumPy array (NOT the deep-learning default!): torch.float64
after explicit .float() cast: torch.float32


## 3. Device detection and movement

🎯 **Interview Tip:** writing device-agnostic code (detect the best available device, then use
it everywhere) is standard practice -- code that only works with a hardcoded `'cuda'` breaks on
any machine without an NVIDIA GPU.

In [4]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():   # Apple Silicon GPU backend
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("best available device on this machine:", device)

cpu_tensor = torch.randn(4, 4)
moved_tensor = cpu_tensor.to(device)
print("original device:", cpu_tensor.device, "-> moved to:", moved_tensor.device)

back_on_cpu = moved_tensor.to("cpu")
assert torch.allclose(cpu_tensor, back_on_cpu)
print("\nvalues survive a round-trip to the accelerated device and back, unchanged.")


best available device on this machine: mps


original device: cpu -> moved to: mps:0

values survive a round-trip to the accelerated device and back, unchanged.


## 4. Device-mismatch error, deliberately triggered

⚠️ **Common Mistake:** operating on two tensors that live on different devices -- PyTorch
raises an error rather than silently doing something unexpected (e.g. silently moving data,
which would hide a performance bug).

In [5]:
cpu_tensor_a = torch.randn(3, 3)
device_tensor_b = torch.randn(3, 3).to(device)

try:
    result = cpu_tensor_a + device_tensor_b
    print("no error raised (both tensors ended up on the same device: CPU-only machine)")
except RuntimeError as e:
    print(f"device mismatch error (expected when device != cpu): {e}")


device mismatch error (expected when device != cpu): Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!


## 5. Indexing, reshape/view, transpose, broadcasting, matmul -- mirrored against NumPy

💡 **Key Idea:** the syntax is nearly identical to Day 1's `02_numpy`; the only new thing is the
device each tensor lives on.

In [6]:
rng = np.random.default_rng(0)
numpy_X = rng.normal(size=(4, 5)).astype(np.float32)
torch_X = torch.from_numpy(numpy_X.copy())

# Indexing
assert np.allclose(numpy_X[1, :], torch_X[1, :].numpy())
assert np.allclose(numpy_X[:, 2], torch_X[:, 2].numpy())

# Reshape / view
numpy_reshaped = numpy_X.reshape(2, 10)
torch_reshaped = torch_X.reshape(2, 10)   # .view() would also work here since torch_X is contiguous
assert np.allclose(numpy_reshaped, torch_reshaped.numpy())

# Transpose
assert np.allclose(numpy_X.T, torch_X.T.numpy())

# Broadcasting
numpy_bias = rng.normal(size=(5,)).astype(np.float32)
torch_bias = torch.from_numpy(numpy_bias)
assert np.allclose(numpy_X + numpy_bias, (torch_X + torch_bias).numpy())

# Matrix multiplication
numpy_W = rng.normal(size=(5, 3)).astype(np.float32)
torch_W = torch.from_numpy(numpy_W)
assert np.allclose(numpy_X @ numpy_W, (torch_X @ torch_W).numpy(), atol=1e-5)

print("every operation's PyTorch result matches its NumPy equivalent.")


every operation's PyTorch result matches its NumPy equivalent.


## 6. `.view()` vs `.reshape()`: the contiguity gotcha

In [7]:
contiguous_tensor = torch.arange(12).reshape(3, 4)
transposed = contiguous_tensor.T   # transpose does NOT copy -- the result is non-contiguous

print("is contiguous_tensor contiguous?", contiguous_tensor.is_contiguous())
print("is transposed contiguous?        ", transposed.is_contiguous())

try:
    transposed.view(12)   # will fail: view requires contiguous memory
    print("view() succeeded (unexpected)")
except RuntimeError as e:
    print(f"view() failed as expected: {type(e).__name__}")

# Both of these fix it:
reshaped_ok = transposed.reshape(12)       # reshape() copies if needed
view_ok = transposed.contiguous().view(12)  # or make it contiguous first, then view
print("\nreshape() works directly:", reshaped_ok.shape)
print("contiguous().view() also works:", view_ok.shape)
assert torch.equal(reshaped_ok, view_ok)


is contiguous_tensor contiguous? True
is transposed contiguous?         False
view() failed as expected: RuntimeError

reshape() works directly: torch.Size([12])
contiguous().view() also works: torch.Size([12])


## 7. Benchmark: matmul on CPU vs the best available device

🔬 **Experiment:** larger matrices show a bigger gap between CPU and an accelerated device
than small ones -- GPUs/MPS have fixed overhead that only pays off at scale.

In [8]:
import time

size = 2000
A_cpu = torch.randn(size, size)
B_cpu = torch.randn(size, size)

start = time.perf_counter()
_ = A_cpu @ B_cpu
cpu_time = time.perf_counter() - start

A_dev = A_cpu.to(device)
B_dev = B_cpu.to(device)
if device.type != "cpu":
    torch.mps.synchronize() if device.type == "mps" else torch.cuda.synchronize()
start = time.perf_counter()
_ = A_dev @ B_dev
if device.type == "mps":
    torch.mps.synchronize()
elif device.type == "cuda":
    torch.cuda.synchronize()
device_time = time.perf_counter() - start

print(f"{size}x{size} matmul -- CPU: {cpu_time*1000:.1f}ms, {device.type}: {device_time*1000:.1f}ms")
if device.type != "cpu":
    print(f"speedup: {cpu_time / device_time:.1f}x")
else:
    print("(no accelerated device available on this machine -- both times are CPU)")


2000x2000 matmul -- CPU: 7.3ms, mps: 231.2ms
speedup: 0.0x


## 8. Connection to what's next

Every tensor operation here works identically once `requires_grad=True` is set — that's
exactly `19_pytorch_autograd`, where PyTorch starts tracking a computational graph over these
same operations to compute gradients automatically.

## 9. Interview questions
1. What three attributes, beyond values, define a PyTorch tensor?
2. Why does PyTorch raise an error on a device mismatch rather than silently resolving it?
3. What's the difference between `.view()` and `.reshape()`, and when does `.view()` fail?
4. Why would you write a `get_device()` helper rather than hardcoding `'cuda'`?
5. Why might a GPU/MPS matmul not be faster than CPU for very small matrices?

## 10. Exercises
1. Write a function that asserts two tensors have the same shape, dtype, AND device before an operation, with a clear error message if not.
2. Benchmark matmul at several sizes (100, 500, 2000, 5000) and plot CPU time vs accelerated-device time.
3. Convert a batch of NumPy image-like arrays (e.g. shape `(32, 3, 64, 64)`) to a PyTorch tensor, move it to the best device, and verify the round trip preserves values.

## 11. Summary

✅ PyTorch tensors extend NumPy's shape/dtype/broadcasting model with device placement;
`float32` is the default; `.view()` needs contiguous memory while `.reshape()` doesn't; writing
device-agnostic code (`get_device()`) is standard practice. Next: `19_pytorch_autograd`.